Configurando o ambiente

In [0]:
%sql
use catalog stack_overgol;

dim_cliente

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_cliente AS
SELECT
    id_cliente,
    nome_cliente,
    sobrenome_cliente,
    email_cliente,
    telefone_cliente,
    ramal_cliente,
    genero_cliente,
    endereco_cliente,
    cidade_cliente,
    estado_cliente,
    pais_cliente,
    origem_cliente,
    FLOOR(DATEDIFF(current_date(), data_nascimento_cliente) / 365) AS idade
FROM silver.clientes;

num_affected_rows,num_inserted_rows


dim_produto

In [0]:
%sql
CREATE OR REPLACE TABLE gold.dim_produto AS
SELECT
    id_produto,
    nome_produto,
    categoria_produto,
    preco_produto,
    fornecedor_produto,
    estoque_produto,
    produto_ativo,
    CASE 
        WHEN preco_produto < 50 THEN 'baixo'
        WHEN preco_produto < 200 THEN 'medio'
        ELSE 'alto'
    END AS faixa_preco
FROM silver.catalogo_produtos;

num_affected_rows,num_inserted_rows


fato_vendas

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_vendas AS
SELECT
    p.id_pedido,
    p.data_pedido,
    p.id_cliente,
    c.nome_cliente,
    p.id_produto,
    pr.nome_produto,
    pr.categoria_produto,
    p.quantidade_produto,
    p.valor_pedido,
    p.metodo_pagamento,
    p.status_pedido
FROM silver.pedidos p
LEFT JOIN gold.dim_cliente c 
    ON p.id_cliente = c.id_cliente
LEFT JOIN gold.dim_produto pr 
    ON p.id_produto = pr.id_produto;

num_affected_rows,num_inserted_rows


fato_suporte

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_suporte AS
SELECT
    s.ticket_id,
    s.id_cliente,
    c.nome_cliente,
    s.id_pedido,
    p.data_pedido,
    s.tipo_problema,
    s.data_abertura,
    s.data_resolucao,
    s.tempo_resolucao_horas,
    s.agente_suporte
FROM silver.suporte_tickets s
LEFT JOIN gold.dim_cliente c 
    ON s.id_cliente = c.id_cliente
LEFT JOIN silver.pedidos p 
    ON s.id_pedido = p.id_pedido;

num_affected_rows,num_inserted_rows


fato_avaliacoes

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_avaliacoes AS
SELECT
    a.id_avaliacao,
    a.id_cliente,
    c.nome_cliente,
    a.id_produto,
    pr.nome_produto,
    a.id_pedido,
    a.nota_produto,
    a.nota_nps,
    a.recomenda_produto,
    a.data_avaliacao
FROM silver.avaliacoes a
LEFT JOIN gold.dim_cliente c 
    ON a.id_cliente = c.id_cliente
LEFT JOIN gold.dim_produto pr 
    ON a.id_produto = pr.id_produto;

num_affected_rows,num_inserted_rows


fato_eventos

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_eventos AS
SELECT
    id_evento,
    id_sessao,
    id_cliente,
    id_produto,
    tipo_evento,
    canal_evento,
    origem_sessao,
    data_evento,
    tempo_pagina_seg
FROM silver.clickstream;

num_affected_rows,num_inserted_rows


fato_360_cliente

In [0]:
%sql
CREATE OR REPLACE TABLE gold.fato_360_cliente AS
SELECT
    c.id_cliente,
    c.nome_cliente,
    COUNT(DISTINCT p.id_pedido) AS total_pedidos,
    SUM(p.valor_pedido) AS total_gasto,
    COUNT(DISTINCT s.ticket_id) AS total_tickets_suporte,
    AVG(a.nota_nps) AS nps_medio,
    AVG(a.nota_produto) AS nota_media_produto
FROM gold.dim_cliente c
LEFT JOIN silver.pedidos p 
    ON c.id_cliente = p.id_cliente
LEFT JOIN silver.suporte_tickets s 
    ON c.id_cliente = s.id_cliente
LEFT JOIN silver.avaliacoes a 
    ON c.id_cliente = a.id_cliente
GROUP BY
    c.id_cliente,
    c.nome_cliente;

num_affected_rows,num_inserted_rows


Z-Order (Otimização)

In [0]:
%sql

OPTIMIZE gold.fato_vendas
ZORDER BY (id_cliente, data_pedido);

OPTIMIZE gold.fato_suporte
ZORDER BY (id_cliente, id_pedido);

OPTIMIZE gold.fato_avaliacoes
ZORDER BY (id_produto, id_cliente);

OPTIMIZE gold.fato_eventos
ZORDER BY (id_cliente, id_sessao);

OPTIMIZE gold.fato_360_cliente
ZORDER BY (id_cliente);

OPTIMIZE gold.dim_cliente
ZORDER BY (id_cliente);

OPTIMIZE gold.dim_produto
ZORDER BY (id_produto);

path,metrics
,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 12611), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1777733025584, 1777733027671, 8, 0, null, List(0, 0), null, 8, 8, 0, 0, null, null)"
